In [7]:
# Quantos jogos cada temporada teve?
df['ano'].value_counts().sort_index()

ano
2003    552
2004    552
2005    462
2006    380
2007    380
2008    380
2009    380
2010    380
2011    380
2012    380
2013    380
2014    380
2015    380
2016    379
2017    380
2018    380
2019    380
2020    268
2021    492
2022    380
2023    380
2024    380
2025    380
Name: count, dtype: int64

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9165 entries, 0 to 9164
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ID                  9165 non-null   int64         
 1   rodata              9165 non-null   int64         
 2   data                9165 non-null   datetime64[us]
 3   hora                9165 non-null   str           
 4   mandante            9165 non-null   str           
 5   visitante           9165 non-null   str           
 6   formacao_mandante   4190 non-null   str           
 7   formacao_visitante  4190 non-null   str           
 8   tecnico_mandante    4555 non-null   str           
 9   tecnico_visitante   4555 non-null   str           
 10  vencedor            9165 non-null   str           
 11  arena               9165 non-null   str           
 12  mandante_Placar     9165 non-null   int64         
 13  visitante_Placar    9165 non-null   int64         
 14  man

In [9]:
import pandas as pd

df = pd.read_csv('campeonato-brasileiro-full.csv')

df.head()

,ID,rodata,data,hora,mandante,visitante,formacao_mandante,formacao_visitante,tecnico_mandante,tecnico_visitante,vencedor,arena,mandante_Placar,visitante_Placar,mandante_Estado,visitante_Estado,arrecadacao
0,1,1,29/03/2003,16:00,Guarani,Vasco,NaN,NaN,NaN,NaN,Guarani,Brinco de Ouro,4,2,SP,RJ,NaN
1,2,1,29/03/2003,16:00,Athletico-PR,Gremio,NaN,NaN,NaN,NaN,Athletico-PR,Arena da Baixada,2,0,PR,RS,NaN
2,3,1,30/03/2003,16:00,Flamengo,Coritiba,NaN,NaN,NaN,NaN,-,Maracanã,1,1,RJ,PR,NaN
3,4,1,30/03/2003,16:00,Goias,Paysandu,NaN,NaN,NaN,NaN,-,Serra Dourada,2,2,GO,PA,NaN
4,5,1,30/03/2003,16:00,Internacional,Ponte Preta,NaN,NaN,NaN,NaN,-,Beira Rio,1,1,RS,SP,NaN


In [10]:
# Converter a coluna de data para o formato de data de verdade
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')

# Criar uma coluna só com o ano, para facilitar análises por temporada
df['ano'] = df['data'].dt.year

df[['data', 'ano']].head()

,data,ano
0,2003-03-29,2003
1,2003-03-29,2003
2,2003-03-30,2003
3,2003-03-30,2003
4,2003-03-30,2003


In [11]:
def classificar_resultado(linha):
    if linha['mandante_Placar'] > linha['visitante_Placar']:
        return 'vitoria_mandante'
    elif linha['mandante_Placar'] < linha['visitante_Placar']:
        return 'vitoria_visitante'
    else:
        return 'empate'

df['resultado'] = df.apply(classificar_resultado, axis=1)

df['resultado'].value_counts()

resultado
vitoria_mandante     4550
empate               2421
vitoria_visitante    2194
Name: count, dtype: int64

In [6]:
df['vencedor'].value_counts().head(10)

vencedor
-                2421
Flamengo          397
Sao Paulo         396
Palmeiras         374
Internacional     368
Corinthians       353
Fluminense        353
Santos            352
Atletico-MG       350
Gremio            341
Name: count, dtype: int64

In [12]:
def calcular_pontos_tabela(df):
    times = pd.concat([df['mandante'], df['visitante']]).unique()
    tabela = []

    for time in times:
        jogos_casa = df[df['mandante'] == time]
        jogos_fora = df[df['visitante'] == time]

        vitorias = len(jogos_casa[jogos_casa['resultado'] == 'vitoria_mandante']) + \
                   len(jogos_fora[jogos_fora['resultado'] == 'vitoria_visitante'])
        empates = len(jogos_casa[jogos_casa['resultado'] == 'empate']) + \
                  len(jogos_fora[jogos_fora['resultado'] == 'empate'])
        derrotas = len(jogos_casa[jogos_casa['resultado'] == 'vitoria_visitante']) + \
                   len(jogos_fora[jogos_fora['resultado'] == 'vitoria_mandante'])

        gols_pro = jogos_casa['mandante_Placar'].sum() + jogos_fora['visitante_Placar'].sum()
        gols_contra = jogos_casa['visitante_Placar'].sum() + jogos_fora['mandante_Placar'].sum()

        jogos = vitorias + empates + derrotas
        pontos = vitorias * 3 + empates

        tabela.append({
            'time': time,
            'jogos': jogos,
            'vitorias': vitorias,
            'empates': empates,
            'derrotas': derrotas,
            'gols_pro': gols_pro,
            'gols_contra': gols_contra,
            'saldo_gols': gols_pro - gols_contra,
            'pontos': pontos
        })

    return pd.DataFrame(tabela).sort_values('pontos', ascending=False).reset_index(drop=True)

# Testando com a temporada de 2022
tabela_2022 = calcular_pontos_tabela(df[df['ano'] == 2022])
tabela_2022.head(10)

,time,jogos,vitorias,empates,derrotas,gols_pro,gols_contra,saldo_gols,pontos
0,Palmeiras,38,23,12,3,66,27,39,81
1,Internacional,38,20,13,5,58,31,27,73
2,Fluminense,38,21,7,10,63,41,22,70
3,Corinthians,38,18,11,9,44,36,8,65
4,Flamengo,38,18,8,12,60,39,21,62
5,Atletico-MG,38,15,13,10,45,37,8,58
6,Athletico-PR,38,16,10,12,48,48,0,58
7,Fortaleza,38,15,10,13,46,39,7,55
8,Sao Paulo,38,13,15,10,55,42,13,54
9,Botafogo-RJ,38,15,8,15,41,43,-2,53
